# Setup

## Data Cleaning

In [ ]:
import pandas as pd

sample = pd.read_parquet("data/movies_sample.parquet")
sample["genre"].head(10).tolist()

In [ ]:
sample["genre"] = sample["genre"].str.split(",")
sample["genre"] = sample["genre"].apply(lambda x : [item.lower().strip() for item in x])
sample["genre"].head(10).tolist()

In [ ]:
# Remove dirty genres
def clean_genres(g):
  out = []
  for x in list(g):
    x = x.strip().lower().removesuffix(" film")
    if x and x not in out: # do not add duplicates
      out.append(x)
  return out

sample["genres"] = sample["genre"].apply(clean_genres)
sample = sample[sample["genres"].str.len() > 0]
sample["genres"].head(10).tolist()

# Qdrant setup

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

client = QdrantClient(url="http://localhost:6333")

COLLECTION_NAME = "movies_chunked"

if not client.collection_exists(collection_name=COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )

print(client.get_collections())


In [ ]:
from qdrant_client.models import PayloadSchemaType


client.create_payload_index(
  collection_name=COLLECTION_NAME,
  field_schema=PayloadSchemaType.INTEGER,
  field_name="year",
)

client.create_payload_index(
  collection_name=COLLECTION_NAME,
  field_schema=PayloadSchemaType.KEYWORD,
  field_name="genres",
)

info = client.get_collection(collection_name=COLLECTION_NAME)

info.payload_schema

## Re-Chunking Plots

In [ ]:
import re

SENT_SPLIT = re.compile(r'(?:(?<=[.!?])|(?<=[.!?]["\']))\s+(?=["\'A-Z0-9])')

tokenizer = model.tokenizer

def n_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def chunk_plot(plot, prefix="", max_tokens=150, overlap=1):
    """Pack whole sentences into chunks of <= max_tokens (prefix included),
    carrying the last `overlap` sentence(s) into the next chunk."""
    sentences = [s.strip() for s in SENT_SPLIT.split(plot) if s.strip()]
    budget = max_tokens - n_tokens(prefix)

    chunks, cur, cur_len = [], [], 0          # cur holds (sentence, token_count)
    for s in sentences:
        l = n_tokens(s)
        if cur and cur_len + l > budget:
            chunks.append(" ".join(x for x, _ in cur))
            cur = cur[-overlap:] if overlap else []
            cur_len = sum(t for _, t in cur)
            if cur and cur_len + l > budget:   # overlap sentence too big, drop it
                cur, cur_len = [], 0
        cur.append((s, l))
        cur_len += l
    if cur:
        chunks.append(" ".join(x for x, _ in cur))
    return chunks

# try it on one film
r = sample.iloc[0]
for i, c in enumerate(chunk_plot(r["plot"], prefix=f'{r["title"]} ({int(r["year"])}): ')):
    print(i, n_tokens(c), c[:120], "...\n")

In [ ]:
rows = []
for film_id, r in enumerate(sample.itertuples(index=False)):
    prefix = f"{r.title} ({int(r.year)}): "
    for idx, text in enumerate(chunk_plot(r.plot, prefix=prefix)):
        rows.append({
            "film_id": film_id,
            "chunk_idx": idx,
            "title": r.title,
            "year": int(r.year),
            "genres": list(r.genres),
            "text": text,
            "embed_text": prefix + text,
        })

chunks = pd.DataFrame(rows)
chunks.shape

In [ ]:
chunks["tok"] = [n_tokens("passage: " + t) for t in chunks["embed_text"]]
print("chunks:", len(chunks), "| chunks per film:")
print(chunks.groupby("film_id").size().describe())
print("\ntokens per chunk:")
print(chunks["tok"].describe())
print("\nover model limit:", (chunks["tok"] > model.max_seq_length).sum())

# coverage: every sentence of the first 200 films appears in some chunk
for fid in range(200):
    joined = " ".join(chunks.loc[chunks.film_id == fid, "text"])
    for s in SENT_SPLIT.split(sample.iloc[fid]["plot"]):
        assert s.strip() in joined, (fid, s[:60])
print("coverage OK")

## Point insertion

In [ ]:
from itertools import islice
for chunk in islice(chunks.itertuples(index=False), 50):
    print(chunk)

In [ ]:
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer

vec_path = Path("data/vectors-movie-chunked.npy")
model = SentenceTransformer("intfloat/multilingual-e5-small")


In [ ]:
if vec_path.exists():
  vectors = np.load(vec_path)
else:
  texts = ["passage: " + p for p in chunks["text"]]
  vectors = model.encode(
    texts,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True
  )
  np.save(vec_path, vectors)

vectors.shape

In [ ]:
from qdrant_client.models import PointStruct


points = [
  PointStruct(
    id=i, 
    vector=vectors[i].tolist(), 
    payload={
      "title": chunk.title, 
      "year" : int(chunk.year), 
      "genres": list(chunk.genres),
      "text": chunk.text,
      "chunk_idx": chunk.chunk_idx,
      "film_id": chunk.film_id
      }
    )
  for i, chunk in enumerate(chunks.itertuples(index=False)) 
]

points[-5:]

In [ ]:
client.upload_points(collection_name=COLLECTION_NAME, points=points, batch_size=256)
client.count(collection_name=COLLECTION_NAME)

In [ ]:
print(client.retrieve(collection_name=COLLECTION_NAME, ids=[0], with_payload=True))

# First Search

In [ ]:
def search(text: str, limit = 5, **kwargs):
  q = model.encode("query: " + text, normalize_embeddings=True).tolist()
  res = client.query_points(COLLECTION_NAME, query=q, limit=limit, with_payload=True, **kwargs)
  return [(p.payload["title"], p.payload["year"], round(p.score, ndigits=2)) for p in res.points]


In [ ]:
print(search("The intrigue continues, with kidnapping, police abuse, assault, and ransom as more characters are murdered, and betrayal within his own household leads to an unexpected conclusion."))

print(search("Private detective and former police officer “Tip O’Neill” is hired by a wealthy aristocrat to solve the shooting of his dog."))

In [ ]:
import random, numpy as np

def search_ids(text, limit = 10):
  q = model.encode("query: " + text, normalize_embeddings=True).tolist()
  return [p.id for p in client.query_points(COLLECTION_NAME, query = q, limit = limit).points]

rng = random.Random()
test_ids = rng.sample(range(len(sample)), 50)

scores = []
for i in test_ids:
  q_text = ". ".join(chunks.iloc[i]["text"].split(". ")[:2])
  found = search_ids(q_text)
  scores.append(1/ (found.index(i) + 1) if i in found else 0)

print("MRR@10", np.mean(scores))

In [ ]:
import random


def mrr_for(make_query, n = 50, seed = 0):
  rng =  random.Random(seed)
  ids = [i for i in range(len(sample)) if len(chunks.iloc[i]["text"].split(". ")) >= 6]
  scores = []
  for i in rng.sample(ids, n):
    sents = chunks.iloc[i]["text"].split(". ")
    found = search_ids(make_query(sents))
    scores.append(1/(found.index(i) + 1) if i in found else 0)
  return np.mean(scores)

print("first 2 sentences:", mrr_for(lambda s: ". ".join(s[:2])))
print("first 1 sentence: ", mrr_for(lambda s: s[0]))
print("middle sentence:  ", mrr_for(lambda s: s[len(s)//2]))
print("last sentence:    ", mrr_for(lambda s: s[-1]))